# PSM prod-extraction-v1 — Colab smoke train

Phase 5 production-memory training on **Hugging Face + Colab** (not RunPod).

- Resume **`058000`** from [`subbu83/psm-50m-mixed-v1-run`](https://huggingface.co/subbu83/psm-50m-mixed-v1-run)
- Curriculum **`prod-extraction-v1`** on model repo `subbu83/psm-50m-mixed-v1-run` (`prod-memory/prod-extraction-v1.jsonl`)
- Smoke: **+2000 steps** (058000 → 078000), LR **2e-5**, prod grounding eval every **500** steps

Runtime: **GPU** (T4/L4/A100).

**HF tokens** — cell **1b** prompts if Colab secrets are missing:
- **Model** `HF_TOKEN` — read on `subbu83/psm-50m-mixed-v1-run` (write too if uploading checkpoints)
- **Dataset** `DATASET_HF_TOKEN` — read on `chkrishna2001/psm-50m-action-mixed-v1` (can be same token)

Create at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).

Optional: Colab **Secrets** (🔑 sidebar) → `HF_TOKEN`, `DATASET_HF_TOKEN`.

## 1. Setup

### Kernel check (run this first in the notebook UI)

Top-right **Select Kernel** → **Colab** → **GPU**. Then run the next cell.

You should see `Colab runtime: True` and a GPU name. **Agent/MCP cannot drive the Colab kernel** with the current `vscode-notebook-mcp-server` — you must run cells yourself (▶ or Shift+Enter).

In [ ]:
import sys
print("python:", sys.executable)
try:
    import google.colab  # noqa: F401
    on_colab = True
except ImportError:
    on_colab = False
print("Colab runtime:", on_colab)

import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
assert on_colab, "Select Colab GPU kernel (top-right), not local Python"
assert torch.cuda.is_available(), "Colab runtime must be GPU"

In [1]:
!pip install -q huggingface_hub numpy
![ -d /content/PSM ] || git clone --depth 1 https://github.com/chkrishna2001/PSM.git /content/PSM
%cd /content/PSM
!git pull --ff-only || true

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\chkri\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


C:\content\PSM


'[' is not recognized as an internal or external command,
operable program or batch file.
Cloning into '/content/PSM'...
Filtering content:  16% (2/12)
Filtering content:  25% (3/12), 226.81 MiB | 55.22 MiB/s
Filtering content:  25% (3/12), 340.22 MiB | 53.51 MiB/s
Filtering content:  33% (4/12), 340.22 MiB | 53.51 MiB/s
Filtering content:  41% (5/12), 340.22 MiB | 53.51 MiB/s
Filtering content:  50% (6/12), 567.03 MiB | 93.31 MiB/s
Filtering content:  58% (7/12), 680.43 MiB | 101.78 MiB/s
Filtering content:  66% (8/12), 680.43 MiB | 101.78 MiB/s
Filtering content:  75% (9/12), 907.24 MiB | 128.93 MiB/s
Filtering content:  83% (10/12), 945.04 MiB | 46.21 MiB/s
Filtering content:  91% (11/12), 1.03 GiB | 20.67 MiB/s  
Filtering content: 100% (12/12), 1.14 GiB | 20.35 MiB/s
Filtering content: 100% (12/12), 1.25 GiB | 11.78 MiB/s, done.


Already up to date.


In [ ]:
import os
from pathlib import Path

def _colab_secret(name: str) -> str | None:
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

def _local_hf_token() -> str:
    for candidate in [
        Path.home() / ".cache" / "huggingface" / "token",
        Path(os.environ.get("USERPROFILE", "")) / ".cache" / "huggingface" / "token",
    ]:
        if candidate.exists():
            return candidate.read_text(encoding="utf-8").strip()
    return ""

def _token(name: str, *, optional: bool = False) -> str:
    value = (os.environ.get(name) or _colab_secret(name) or "").strip()
    if not value and name in {"HF_TOKEN", "DATASET_HF_TOKEN"}:
        value = _local_hf_token()
    if value:
        return value
    if optional:
        return ""
    raise ValueError(
        f"{name} missing. Add Colab secret '{name}' or set os.environ['{name}'] before this cell."
    )

os.environ["HF_TOKEN"] = _token("HF_TOKEN")
dataset_token = _token("DATASET_HF_TOKEN", optional=True)
os.environ["DATASET_HF_TOKEN"] = dataset_token or os.environ["HF_TOKEN"]
os.environ["PYTHONPATH"] = "psm-model/src:psm-model/prod-memory"
print("HF_TOKEN set:", bool(os.environ["HF_TOKEN"]))
print("DATASET_HF_TOKEN set:", bool(os.environ["DATASET_HF_TOKEN"]))

HF_TOKEN set: True
DATASET_HF_TOKEN set: True


## 2. CUDA check

In [2]:
import json, torch
print(json.dumps({
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}, indent=2))
assert torch.cuda.is_available(), "Switch runtime to GPU."

{
  "torch": "2.11.0+cu128",
  "cuda_available": true,
  "device": "Tesla T4"
}


## 3. Download HF assets (058000 + curriculum + probes)

In [1]:
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path("/content/PSM")
if not ROOT.exists():
    raise FileNotFoundError("Run the Setup cell first to clone the repo into /content/PSM")
os.chdir(ROOT)
env = {
    **os.environ,
    "PYTHONPATH": os.pathsep.join([
        str(ROOT / "psm-model/src"),
        str(ROOT / "psm-model/prod-memory"),
    ]),
}
subprocess.run(
    [sys.executable, "-m", "prod_memory.colab_sync", "download", "--root", str(ROOT)],
    env=env,
    check=True,
)
print("download root:", ROOT)

CalledProcessError: Command '['/usr/bin/python3', '-m', 'prod_memory.colab_sync', 'download', '--root', '/content']' returned non-zero exit status 1.

### 3b. Curriculum missing? Build + upload to model repo

Run this **only if** cell 3a failed with `prod-extraction-v1.jsonl` 404. Uses your `HF_TOKEN` (write) on `subbu83/psm-50m-mixed-v1-run`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from huggingface_hub import HfApi

from prod_memory.hf_assets import CURRICULUM_REL, DEFAULT_MODEL_REPO, MANIFEST_REL

root = Path("/content/PSM")
curriculum = root / CURRICULUM_REL
manifest = root / MANIFEST_REL

if not curriculum.exists():
    direct = root / "psm-model/data/probes/direct_probes.jsonl"
    cmd = [sys.executable, "-m", "prod_memory.build_prod_extraction_v1", "--output", str(curriculum)]
    if direct.exists():
        cmd.extend(["--direct-probes", str(direct)])
    print("Building curriculum:", " ".join(cmd))
    subprocess.run(cmd, cwd=root, check=True, env={**dict(os.environ), "PYTHONPATH": "psm-model/src:psm-model/prod-memory"})

api = HfApi(token=os.environ["HF_TOKEN"])
for remote, path in [(CURRICULUM_REL, curriculum), (MANIFEST_REL, manifest)]:
    if path.exists():
        api.upload_file(
            path_or_fileobj=str(path),
            path_in_repo=remote,
            repo_id=DEFAULT_MODEL_REPO,
            repo_type="model",
            commit_message=f"prod-memory: sync {remote}",
        )
        print("uploaded", remote, path.stat().st_size, "bytes")

In [ ]:
from pathlib import Path
candidates = {
    "checkpoint": ["psm-model/checkpoints/real-v3-50m-full-v2-step-058000.pt"],
    "curriculum": ["prod-memory/prod-extraction-v1.jsonl"],
    "expanded_probe": [
        "data/direct-behavior-v1/expanded-probe-v1-filtered.jsonl",
        "probes/expanded-probe-v1-filtered.jsonl",
    ],
    "direct_probe": ["data/probes/direct_probes.jsonl", "probes/direct_probes.jsonl"],
}
resolved = {}
missing = []
for key, paths in candidates.items():
    hit = next((p for p in paths if Path(p).exists()), None)
    if hit:
        resolved[key] = hit
    else:
        missing.append(key)
print("resolved:", resolved)
print("missing:", missing)
assert not missing

## 4. Train (+2000 steps from 058000)

In [ ]:
RESUME = "psm-model/checkpoints/real-v3-50m-full-v2-step-058000.pt"
TOK = "psm-model/checkpoints/real-v3-50m-full-v2-step-058000.tokenizer.json"
CURRICULUM = "prod-memory/prod-extraction-v1.jsonl"
TARGET_STEPS = 78000
BATCH_SIZE = 8  # use 4 if OOM
LEARNING_RATE = 2e-5

!PYTHONPATH=psm-model/src python -m psm_model.train \
  {CURRICULUM} \
  --out psm-model/checkpoints/real-v3-50m-full-v2.pt \
  --resume {RESUME} \
  --tokenizer {TOK} \
  --steps {TARGET_STEPS} \
  --batch-size {BATCH_SIZE} \
  --learning-rate {LEARNING_RATE} \
  --min-learning-rate 5e-6 \
  --warmup-steps 50 \
  --preset 50m \
  --output-format tagged \
  --device cuda \
  --save-every 500 \
  --metrics-out psm-model/checkpoints/real-v3-50m-full-v2-prod-extraction.metrics.jsonl \
  --structural-loss-weight 1 \
  --eval-every 500 \
  --probe {resolved['expanded_probe']} \
  --manual-probe {resolved['direct_probe']} \
  --abort-after-step 90000 \
  --collapse-threshold 0.90

## 5. Prod grounding eval (Phase 1 fixtures)

In [ ]:
import json
from pathlib import Path

ckpt_dir = Path("psm-model/checkpoints")
eval_steps = sorted(int(p.stem.split("-")[-1]) for p in ckpt_dir.glob("real-v3-50m-full-v2-step-*.pt"))
print("checkpoints:", eval_steps[-5:])

for step in eval_steps:
    if step < 58500 or step > TARGET_STEPS:
        continue
    ckpt = ckpt_dir / f"real-v3-50m-full-v2-step-{step:06d}.pt"
    out = Path(f"psm-model/prod-memory/results/prod-grounding-step-{step:06d}.json")
    out.parent.mkdir(parents=True, exist_ok=True)
    !PYTHONPATH=psm-model/src:psm-model/prod-memory python -m prod_memory.eval_grounding \
      --checkpoint {ckpt} \
      --checkpoint-label {step} \
      --device cuda \
      --out {out}

## 6. Upload step checkpoints to HF (optional)

In [ ]:
UPLOAD_STEPS = [s for s in eval_steps if s > 58000]
print("upload steps:", UPLOAD_STEPS)
!PYTHONPATH=psm-model/src:psm-model/prod-memory python -m prod_memory.colab_sync upload-steps \
  --root /content/PSM \
  --steps {' '.join(str(s) for s in UPLOAD_STEPS)}

## 7. Download results locally

From Colab: **Files → download** `psm-model/prod-memory/results/prod-grounding-step-*.json` and metrics JSONL.